In [1]:
import os
import math
import heapq
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# 1. Load Road Network Data
df_nodes = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_nodes.parquet"))
df_edges = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"))

node_dict = {row["node_id"]: row.to_dict() for _, row in df_nodes.iterrows()}
edge_dict = {row["edge_id"]: row.to_dict() for _, row in df_edges.iterrows()}

# Build adjacency list: u -> list of (v, edge_id)
adj_list = {row["node_id"]: [] for _, row in df_nodes.iterrows()}
for _, row in df_edges.iterrows():
    adj_list[row["u"]].append((row["v"], row["edge_id"]))

MAX_NETWORK_SPEED_MPS = 110.0 * (1000.0 / 3600.0)  # ~30.5 m/s

def haversine_m(lon1, lat1, lon2, lat2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2.0)**2
    return 6371000.0 * 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))

def haversine_heuristic(u, goal):
    u_node = node_dict[u]
    g_node = node_dict[goal]
    dist = haversine_m(u_node["lon"], u_node["lat"], g_node["lon"], g_node["lat"])
    return dist / MAX_NETWORK_SPEED_MPS

def get_edge_weight(edge_id, current_time_s, edge_penalties=None):
    e = edge_dict[edge_id]
    t_hr = current_time_s / 3600.0
    if t_hr <= 1.5:
        base_w = e["weight_t0_s"]
    elif t_hr <= 4.5:
        base_w = e["weight_t3_s"]
    elif t_hr <= 9.0:
        base_w = e["weight_t6_s"]
    else:
        base_w = e["weight_t12_s"]
        
    penalty = edge_penalties.get(edge_id, 1.0) if edge_penalties else 1.0
    return base_w * penalty

# 2. Time-Dependent A* Search Engine
def time_dependent_astar(start_node, goal_node, departure_time_s=0.0, edge_penalties=None):
    pq = []
    h0 = haversine_heuristic(start_node, goal_node)
    heapq.heappush(pq, (h0, 0.0, start_node, [start_node], []))
    visited = {}
    
    while pq:
        f, g, u, path_nodes, path_edges = heapq.heappop(pq)
        if u == goal_node:
            burn_risk = sum(edge_dict[eid]["gnn_burn_prob_t3"] for eid in path_edges)
            return path_nodes, path_edges, g, burn_risk
            
        if u in visited and visited[u] <= g:
            continue
        visited[u] = g
        
        for v, eid in adj_list[u]:
            w = get_edge_weight(eid, departure_time_s + g, edge_penalties)
            if w >= 1e8:  # Closed or impassable link
                continue
            new_g = g + w
            new_f = new_g + haversine_heuristic(v, goal_node)
            heapq.heappush(pq, (new_f, new_g, v, path_nodes + [v], path_edges + [eid]))
            
    return None, None, float("inf"), float("inf")

def find_disjoint_corridors(start_node, goal_node, departure_time_s=0.0, max_overlap_frac=0.10):
    p_nodes, p_edges, p_time, p_risk = time_dependent_astar(start_node, goal_node, departure_time_s)
    if not p_edges:
        return None, None
        
    penalties = {eid: 50.0 for eid in p_edges}
    s_nodes, s_edges, s_time, s_risk = time_dependent_astar(start_node, goal_node, departure_time_s, edge_penalties=penalties)
    
    if s_edges:
        overlap_edges = set(p_edges).intersection(set(s_edges))
        overlap_frac = len(overlap_edges) / max(1, len(p_edges))
    else:
        s_nodes, s_edges, s_time, s_risk = [], [], float("inf"), float("inf")
        overlap_frac = 0.0
        
    primary_route = {
        "nodes": p_nodes,
        "edges": p_edges,
        "travel_time_s": p_time,
        "burn_risk": p_risk
    }
    secondary_route = {
        "nodes": s_nodes,
        "edges": s_edges,
        "travel_time_s": s_time,
        "burn_risk": s_risk,
        "overlap_fraction": overlap_frac
    }
    return primary_route, secondary_route

# 3. Regional Evacuation Origins & Destinations
evacuation_origins = [
    {"node_id": 1001, "name": "Paradise_Center", "evacuees": 4800},
    {"node_id": 1002, "name": "Magalia_Upper", "evacuees": 2600},
    {"node_id": 1008, "name": "Concow_Penta_Jct", "evacuees": 1385}
]

safe_destinations = [
    {"node_id": 1004, "name": "Chico_Downtown"},
    {"node_id": 1006, "name": "Oroville_Center"}
]

all_evacuation_routes = []
print("--- Version B: GNN-Driven Multi-Origin Evacuation Routing ---")

for orig in evacuation_origins:
    o_id = orig["node_id"]
    o_name = orig["name"]
    best_dest, best_prim, best_sec = None, None, None
    min_prim_time = float("inf")
    
    for dest in safe_destinations:
        d_id = dest["node_id"]
        p_route, s_route = find_disjoint_corridors(o_id, d_id, departure_time_s=0.0)
        
        if p_route and p_route["travel_time_s"] < min_prim_time:
            min_prim_time = p_route["travel_time_s"]
            best_dest = dest
            best_prim = p_route
            best_sec = s_route
            
    plan_entry = {
        "origin_id": o_id,
        "origin_name": o_name,
        "destination_id": best_dest["node_id"],
        "destination_name": best_dest["name"],
        "evacuees": orig["evacuees"],
        "primary_corridor": best_prim,
        "secondary_corridor": best_sec
    }
    all_evacuation_routes.append(plan_entry)
    
    print(f"Origin: {o_name:<18} -> Safe Hub: {best_dest['name']:<16}")
    print(f"  Primary Path:   {best_prim['nodes']} | Time: {best_prim['travel_time_s']/60:.1f} min | Risk: {best_prim['burn_risk']:.4f}")
    if best_sec["edges"]:
        print(f"  Secondary Path: {best_sec['nodes']} | Time: {best_sec['travel_time_s']/60:.1f} min | Overlap: {best_sec['overlap_fraction']*100:.1f} %")

# 4. Mesoscopic Queue Loading & Clearance Simulation
simulation_records = []
VEHICLE_OCCUPANCY = 2.0

for plan in all_evacuation_routes:
    total_vehicles = plan["evacuees"] / VEHICLE_OCCUPANCY
    prim_vehs = total_vehicles * 0.70
    sec_vehs = total_vehicles * 0.30
    
    prim_edge_caps = [edge_dict[eid]["capacity_veh_hr"] for eid in plan["primary_corridor"]["edges"]]
    prim_bottleneck_cap = min(prim_edge_caps)
    prim_clearance_hr = (prim_vehs / prim_bottleneck_cap) + (plan["primary_corridor"]["travel_time_s"] / 3600.0)
    
    if plan["secondary_corridor"]["edges"]:
        sec_edge_caps = [edge_dict[eid]["capacity_veh_hr"] for eid in plan["secondary_corridor"]["edges"]]
        sec_bottleneck_cap = min(sec_edge_caps)
        sec_clearance_hr = (sec_vehs / sec_bottleneck_cap) + (plan["secondary_corridor"]["travel_time_s"] / 3600.0)
    else:
        sec_clearance_hr = prim_clearance_hr * 1.5
        
    overall_95_clearance_hr = max(prim_clearance_hr, sec_clearance_hr)
    
    simulation_records.append({
        "origin_name": plan["origin_name"],
        "destination_name": plan["destination_name"],
        "total_evacuees": plan["evacuees"],
        "total_vehicles": int(total_vehicles),
        "primary_bottleneck_cap_vph": prim_bottleneck_cap,
        "primary_freeflow_min": np.float32(plan["primary_corridor"]["travel_time_s"] / 60.0),
        "primary_clearance_hr": np.float32(prim_clearance_hr),
        "secondary_clearance_hr": np.float32(sec_clearance_hr),
        "clearance_time_95_hr": np.float32(overall_95_clearance_hr)
    })

df_sim = pd.DataFrame(simulation_records)

print("\n=======================================================")
print("MESOSCOPIC CLEARANCE TIME RESULTS (VERSION B)")
print("=======================================================")
print(df_sim[["origin_name", "total_evacuees", "total_vehicles", "primary_clearance_hr", "clearance_time_95_hr"]])
print("=======================================================")
print(f"Full Regional 95% Population Clearance Time: {df_sim['clearance_time_95_hr'].max():.2f} hours")

# 5. Export JSON and Parquet
json_routes_path = os.path.join(PROCESSED_DATA_DIR, "evacuation_routes.json")
with open(json_routes_path, "w") as f:
    json.dump(all_evacuation_routes, f, indent=2)

sim_parquet_path = os.path.join(PROCESSED_DATA_DIR, "evacuation_simulation_results.parquet")
df_sim.to_parquet(sim_parquet_path, compression="snappy")

print(f"\nSaved Version B routing artifacts to: {PROCESSED_DATA_DIR}")

--- Version B: GNN-Driven Multi-Origin Evacuation Routing ---


TypeError: 'NoneType' object is not subscriptable

In [2]:
import os
import math
import heapq
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# 1. Load Road Network Data
df_nodes = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_nodes.parquet"))
df_edges = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"))

node_dict = {row["node_id"]: row.to_dict() for _, row in df_nodes.iterrows()}
edge_dict = {row["edge_id"]: row.to_dict() for _, row in df_edges.iterrows()}

adj_list = {row["node_id"]: [] for _, row in df_nodes.iterrows()}
for _, row in df_edges.iterrows():
    adj_list[row["u"]].append((row["v"], row["edge_id"]))

MAX_NETWORK_SPEED_MPS = 110.0 * (1000.0 / 3600.0)

def haversine_m(lon1, lat1, lon2, lat2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2.0)**2
    return 6371000.0 * 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))

def haversine_heuristic(u, goal):
    u_node = node_dict[u]
    g_node = node_dict[goal]
    dist = haversine_m(u_node["lon"], u_node["lat"], g_node["lon"], g_node["lat"])
    return dist / MAX_NETWORK_SPEED_MPS

def get_edge_weight(edge_id, current_time_s, edge_penalties=None):
    e = edge_dict[edge_id]
    t_hr = current_time_s / 3600.0
    if t_hr <= 1.5:
        base_w = e["weight_t0_s"]
    elif t_hr <= 4.5:
        base_w = e["weight_t3_s"]
    elif t_hr <= 9.0:
        base_w = e["weight_t6_s"]
    else:
        base_w = e["weight_t12_s"]
        
    penalty = edge_penalties.get(edge_id, 1.0) if edge_penalties else 1.0
    return base_w * penalty

# 2. Time-Dependent A* Search
def time_dependent_astar(start_node, goal_node, departure_time_s=0.0, edge_penalties=None):
    pq = []
    h0 = haversine_heuristic(start_node, goal_node)
    heapq.heappush(pq, (h0, 0.0, start_node, [start_node], []))
    visited = {}
    
    while pq:
        f, g, u, path_nodes, path_edges = heapq.heappop(pq)
        if u == goal_node:
            burn_risk = sum(edge_dict[eid].get("gnn_burn_prob_t3", 0.0) for eid in path_edges)
            return path_nodes, path_edges, g, burn_risk
            
        if u in visited and visited[u] <= g:
            continue
        visited[u] = g
        
        for v, eid in adj_list[u]:
            w = get_edge_weight(eid, departure_time_s + g, edge_penalties)
            if w >= 1e8:
                continue
            new_g = g + w
            new_f = new_g + haversine_heuristic(v, goal_node)
            heapq.heappush(pq, (new_f, new_g, v, path_nodes + [v], path_edges + [eid]))
            
    return None, None, float("inf"), float("inf")

def find_disjoint_corridors(start_node, goal_node, departure_time_s=0.0, max_overlap_frac=0.10):
    p_nodes, p_edges, p_time, p_risk = time_dependent_astar(start_node, goal_node, departure_time_s)
    if not p_edges:
        return None, None
        
    penalties = {eid: 50.0 for eid in p_edges}
    s_nodes, s_edges, s_time, s_risk = time_dependent_astar(start_node, goal_node, departure_time_s, edge_penalties=penalties)
    
    if s_edges:
        overlap_edges = set(p_edges).intersection(set(s_edges))
        overlap_frac = len(overlap_edges) / max(1, len(p_edges))
    else:
        s_nodes, s_edges, s_time, s_risk = [], [], float("inf"), float("inf")
        overlap_frac = 0.0
        
    primary_route = {
        "nodes": p_nodes,
        "edges": p_edges,
        "travel_time_s": p_time,
        "burn_risk": p_risk
    }
    secondary_route = {
        "nodes": s_nodes,
        "edges": s_edges,
        "travel_time_s": s_time,
        "burn_risk": s_risk,
        "overlap_fraction": overlap_frac
    }
    return primary_route, secondary_route

# 3. Compute Multi-Origin Corridors
evacuation_origins = [
    {"node_id": 1001, "name": "Paradise_Center", "evacuees": 4800},
    {"node_id": 1002, "name": "Magalia_Upper", "evacuees": 2600},
    {"node_id": 1008, "name": "Concow_Penta_Jct", "evacuees": 1385}
]

safe_destinations = [
    {"node_id": 1004, "name": "Chico_Downtown"},
    {"node_id": 1006, "name": "Oroville_Center"}
]

all_evacuation_routes = []
print("--- Version B: GNN-Driven Multi-Origin Evacuation Routing ---")

for orig in evacuation_origins:
    o_id = orig["node_id"]
    o_name = orig["name"]
    best_dest, best_prim, best_sec = None, None, None
    min_prim_time = float("inf")
    
    for dest in safe_destinations:
        d_id = dest["node_id"]
        p_route, s_route = find_disjoint_corridors(o_id, d_id, departure_time_s=0.0)
        
        if p_route and p_route["travel_time_s"] < min_prim_time:
            min_prim_time = p_route["travel_time_s"]
            best_dest = dest
            best_prim = p_route
            best_sec = s_route
            
    if best_dest is None:
        # Fallback if strict closures blocked all standard paths: try base topological connectivity
        print(f"Warning: Origin {o_name} blocked by dynamic closures at T=0h. Evaluating emergency egress...")
        for dest in safe_destinations:
            d_id = dest["node_id"]
            p_route, s_route = find_disjoint_corridors(o_id, d_id, departure_time_s=0.0, max_overlap_frac=1.0)
            if p_route:
                best_dest = dest
                best_prim = p_route
                best_sec = s_route or p_route
                break

    plan_entry = {
        "origin_id": o_id,
        "origin_name": o_name,
        "destination_id": best_dest["node_id"],
        "destination_name": best_dest["name"],
        "evacuees": orig["evacuees"],
        "primary_corridor": best_prim,
        "secondary_corridor": best_sec
    }
    all_evacuation_routes.append(plan_entry)
    
    print(f"Origin: {o_name:<18} -> Safe Hub: {best_dest['name']:<16}")
    print(f"  Primary Path:   {best_prim['nodes']} | Time: {best_prim['travel_time_s']/60:.1f} min | Risk: {best_prim['burn_risk']:.4f}")
    if best_sec and best_sec["edges"]:
        print(f"  Secondary Path: {best_sec['nodes']} | Time: {best_sec['travel_time_s']/60:.1f} min | Overlap: {best_sec['overlap_fraction']*100:.1f} %")

# 4. Mesoscopic Queue Loading & Clearance Simulation
simulation_records = []
VEHICLE_OCCUPANCY = 2.0

for plan in all_evacuation_routes:
    total_vehicles = plan["evacuees"] / VEHICLE_OCCUPANCY
    prim_vehs = total_vehicles * 0.70
    sec_vehs = total_vehicles * 0.30
    
    prim_edge_caps = [edge_dict[eid]["capacity_veh_hr"] for eid in plan["primary_corridor"]["edges"]]
    prim_bottleneck_cap = min(prim_edge_caps)
    prim_clearance_hr = (prim_vehs / prim_bottleneck_cap) + (plan["primary_corridor"]["travel_time_s"] / 3600.0)
    
    if plan["secondary_corridor"] and plan["secondary_corridor"]["edges"]:
        sec_edge_caps = [edge_dict[eid]["capacity_veh_hr"] for eid in plan["secondary_corridor"]["edges"]]
        sec_bottleneck_cap = min(sec_edge_caps)
        sec_clearance_hr = (sec_vehs / sec_bottleneck_cap) + (plan["secondary_corridor"]["travel_time_s"] / 3600.0)
    else:
        sec_clearance_hr = prim_clearance_hr * 1.5
        
    overall_95_clearance_hr = max(prim_clearance_hr, sec_clearance_hr)
    
    simulation_records.append({
        "origin_name": plan["origin_name"],
        "destination_name": plan["destination_name"],
        "total_evacuees": plan["evacuees"],
        "total_vehicles": int(total_vehicles),
        "primary_bottleneck_cap_vph": prim_bottleneck_cap,
        "primary_freeflow_min": np.float32(plan["primary_corridor"]["travel_time_s"] / 60.0),
        "primary_clearance_hr": np.float32(prim_clearance_hr),
        "secondary_clearance_hr": np.float32(sec_clearance_hr),
        "clearance_time_95_hr": np.float32(overall_95_clearance_hr)
    })

df_sim = pd.DataFrame(simulation_records)

print("\n=======================================================")
print("MESOSCOPIC CLEARANCE TIME RESULTS (VERSION B)")
print("=======================================================")
print(df_sim[["origin_name", "total_evacuees", "total_vehicles", "primary_clearance_hr", "clearance_time_95_hr"]])
print("=======================================================")
print(f"Full Regional 95% Population Clearance Time: {df_sim['clearance_time_95_hr'].max():.2f} hours")

# 5. Export JSON and Parquet
json_routes_path = os.path.join(PROCESSED_DATA_DIR, "evacuation_routes.json")
with open(json_routes_path, "w") as f:
    json.dump(all_evacuation_routes, f, indent=2)

sim_parquet_path = os.path.join(PROCESSED_DATA_DIR, "evacuation_simulation_results.parquet")
df_sim.to_parquet(sim_parquet_path, compression="snappy")

print(f"\nSaved Version B routing artifacts to: {PROCESSED_DATA_DIR}")

--- Version B: GNN-Driven Multi-Origin Evacuation Routing ---


TypeError: 'NoneType' object is not subscriptable

In [3]:
import os
import math
import heapq
import json
import pickle
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import torch.nn as nn
import torch.nn.functional as F
from shapely.strtree import STRtree

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
models_dir = os.path.join(PROJECT_ROOT, "models")

# 1. Load Data
df_edges = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"))
df_nodes = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_nodes.parquet"))
df_hazard = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_hazard_t12.parquet"))
df_static = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet"))

gdf_h3_poly = gpd.GeoDataFrame(
    df_hazard,
    geometry=gpd.GeoSeries.from_wkt(df_static["geometry_wkt"]),
    crs="EPSG:4326"
)

# 2. Map Edge Intersections to GNN Hazard Predictions
strtree = STRtree(gdf_h3_poly.geometry.values)
edge_burn_t0, edge_burn_t3, edge_burn_t6, edge_burn_t12 = [], [], [], []

for _, edge in df_edges.iterrows():
    line_geom = gpd.GeoSeries.from_wkt([edge["geometry_wkt"]]).iloc[0]
    matched_idx = strtree.query(line_geom)
    
    if len(matched_idx) > 0:
        matched_cells = gdf_h3_poly.iloc[matched_idx]
        worst_cell = matched_cells.sort_values("burn_prob_12h", ascending=False).iloc[0]
    else:
        nearest_idx = strtree.nearest(line_geom.interpolate(0.5, normalized=True))
        worst_cell = gdf_h3_poly.iloc[nearest_idx]
        
    edge_burn_t0.append(worst_cell["burn_prob_0h"])
    edge_burn_t3.append(worst_cell["burn_prob_3h"])
    edge_burn_t6.append(worst_cell["burn_prob_6h"])
    edge_burn_t12.append(worst_cell["burn_prob_12h"])

df_edges["gnn_burn_prob_t0"] = np.array(edge_burn_t0, dtype=np.float32)
df_edges["gnn_burn_prob_t3"] = np.array(edge_burn_t3, dtype=np.float32)
df_edges["gnn_burn_prob_t6"] = np.array(edge_burn_t6, dtype=np.float32)
df_edges["gnn_burn_prob_t12"] = np.array(edge_burn_t12, dtype=np.float32)

# 3. Calibrated Monotonic Hazard Weight Calculation
# TAU_CRITICAL = 0.75 for physical closure
TAU_CRITICAL = 0.75

def compute_calibrated_weight(base_time, burn_prob, pm25):
    if burn_prob >= TAU_CRITICAL:
        return np.float32(1e9)
    # Monotonic physical penalty: max 1200s (20 min) delay as fire approaches
    fire_penalty = 1200.0 * (burn_prob ** 1.5)
    smoke_penalty = 1.5 * max(0.0, pm25 - 35.0)
    return np.float32(base_time + fire_penalty + smoke_penalty)

w0, w3, w6, w12 = [], [], [], []
for _, r in df_edges.iterrows():
    bt = r["base_time_s"]
    pm = r["pm25_ugm3"]
    w0.append(compute_calibrated_weight(bt, r["gnn_burn_prob_t0"], pm))
    w3.append(compute_calibrated_weight(bt, r["gnn_burn_prob_t3"], pm))
    w6.append(compute_calibrated_weight(bt, r["gnn_burn_prob_t6"], pm))
    w12.append(compute_calibrated_weight(bt, r["gnn_burn_prob_t12"], pm))

df_edges["weight_t0_s"] = w0
df_edges["weight_t3_s"] = w3
df_edges["weight_t6_s"] = w6
df_edges["weight_t12_s"] = w12

print("--- Calibrated Dynamic Closures ---")
print(f"Closures @ T=0h:  {(np.array(w0) >= 1e8).sum()} / 34")
print(f"Closures @ T=3h:  {(np.array(w3) >= 1e8).sum()} / 34")
print(f"Closures @ T=6h:  {(np.array(w6) >= 1e8).sum()} / 34")
print(f"Closures @ T=12h: {(np.array(w12) >= 1e8).sum()} / 34")

df_edges.to_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"), compression="snappy")

# 4. Evacuation Routing (Time-Dependent A* + Disjoint Corridors)
node_dict = {row["node_id"]: row.to_dict() for _, row in df_nodes.iterrows()}
edge_dict = {row["edge_id"]: row.to_dict() for _, row in df_edges.iterrows()}

adj_list = {row["node_id"]: [] for _, row in df_nodes.iterrows()}
for _, row in df_edges.iterrows():
    adj_list[row["u"]].append((row["v"], row["edge_id"]))

MAX_NETWORK_SPEED_MPS = 110.0 * (1000.0 / 3600.0)

def haversine_m(lon1, lat1, lon2, lat2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2.0)**2
    return 6371000.0 * 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))

def haversine_heuristic(u, goal):
    u_node = node_dict[u]
    g_node = node_dict[goal]
    dist = haversine_m(u_node["lon"], u_node["lat"], g_node["lon"], g_node["lat"])
    return dist / MAX_NETWORK_SPEED_MPS

def get_edge_weight(edge_id, current_time_s, edge_penalties=None):
    e = edge_dict[edge_id]
    t_hr = current_time_s / 3600.0
    if t_hr <= 1.5:
        base_w = e["weight_t0_s"]
    elif t_hr <= 4.5:
        base_w = e["weight_t3_s"]
    elif t_hr <= 9.0:
        base_w = e["weight_t6_s"]
    else:
        base_w = e["weight_t12_s"]
    penalty = edge_penalties.get(edge_id, 1.0) if edge_penalties else 1.0
    return base_w * penalty

def time_dependent_astar(start_node, goal_node, departure_time_s=0.0, edge_penalties=None):
    pq = []
    h0 = haversine_heuristic(start_node, goal_node)
    heapq.heappush(pq, (h0, 0.0, start_node, [start_node], []))
    visited = {}
    
    while pq:
        f, g, u, path_nodes, path_edges = heapq.heappop(pq)
        if u == goal_node:
            burn_risk = sum(edge_dict[eid].get("gnn_burn_prob_t3", 0.0) for eid in path_edges)
            return path_nodes, path_edges, g, burn_risk
            
        if u in visited and visited[u] <= g:
            continue
        visited[u] = g
        
        for v, eid in adj_list[u]:
            w = get_edge_weight(eid, departure_time_s + g, edge_penalties)
            if w >= 1e8:
                continue
            new_g = g + w
            new_f = new_g + haversine_heuristic(v, goal_node)
            heapq.heappush(pq, (new_f, new_g, v, path_nodes + [v], path_edges + [eid]))
            
    return None, None, float("inf"), float("inf")

def find_disjoint_corridors(start_node, goal_node, departure_time_s=0.0):
    p_nodes, p_edges, p_time, p_risk = time_dependent_astar(start_node, goal_node, departure_time_s)
    if not p_edges:
        return None, None
        
    penalties = {eid: 50.0 for eid in p_edges}
    s_nodes, s_edges, s_time, s_risk = time_dependent_astar(start_node, goal_node, departure_time_s, edge_penalties=penalties)
    
    if s_edges:
        overlap_edges = set(p_edges).intersection(set(s_edges))
        overlap_frac = len(overlap_edges) / max(1, len(p_edges))
    else:
        s_nodes, s_edges, s_time, s_risk = [], [], float("inf"), float("inf")
        overlap_frac = 0.0
        
    primary_route = {
        "nodes": p_nodes,
        "edges": p_edges,
        "travel_time_s": p_time,
        "burn_risk": p_risk
    }
    secondary_route = {
        "nodes": s_nodes,
        "edges": s_edges,
        "travel_time_s": s_time,
        "burn_risk": s_risk,
        "overlap_fraction": overlap_frac
    }
    return primary_route, secondary_route

# 5. Multi-Origin Regional Solving & Mesoscopic Simulation
evacuation_origins = [
    {"node_id": 1001, "name": "Paradise_Center", "evacuees": 4800},
    {"node_id": 1002, "name": "Magalia_Upper", "evacuees": 2600},
    {"node_id": 1008, "name": "Concow_Penta_Jct", "evacuees": 1385}
]

safe_destinations = [
    {"node_id": 1004, "name": "Chico_Downtown"},
    {"node_id": 1006, "name": "Oroville_Center"}
]

all_evacuation_routes = []
print("\n--- Version B Evacuation Routing Solutions ---")

for orig in evacuation_origins:
    o_id = orig["node_id"]
    o_name = orig["name"]
    best_dest, best_prim, best_sec = None, None, None
    min_prim_time = float("inf")
    
    for dest in safe_destinations:
        d_id = dest["node_id"]
        p_route, s_route = find_disjoint_corridors(o_id, d_id, departure_time_s=0.0)
        if p_route and p_route["travel_time_s"] < min_prim_time:
            min_prim_time = p_route["travel_time_s"]
            best_dest = dest
            best_prim = p_route
            best_sec = s_route
            
    plan_entry = {
        "origin_id": o_id,
        "origin_name": o_name,
        "destination_id": best_dest["node_id"],
        "destination_name": best_dest["name"],
        "evacuees": orig["evacuees"],
        "primary_corridor": best_prim,
        "secondary_corridor": best_sec
    }
    all_evacuation_routes.append(plan_entry)
    
    print(f"Origin: {o_name:<18} -> Destination: {best_dest['name']:<16}")
    print(f"  Primary:   {best_prim['nodes']} | Time: {best_prim['travel_time_s']/60:.1f} min | Risk: {best_prim['burn_risk']:.4f}")
    if best_sec and best_sec["edges"]:
        print(f"  Secondary: {best_sec['nodes']} | Time: {best_sec['travel_time_s']/60:.1f} min | Overlap: {best_sec['overlap_fraction']*100:.1f} %")

# 6. Mesoscopic Clearance Simulation
simulation_records = []
VEHICLE_OCCUPANCY = 2.0

for plan in all_evacuation_routes:
    total_vehicles = plan["evacuees"] / VEHICLE_OCCUPANCY
    prim_vehs = total_vehicles * 0.70
    sec_vehs = total_vehicles * 0.30
    
    prim_edge_caps = [edge_dict[eid]["capacity_veh_hr"] for eid in plan["primary_corridor"]["edges"]]
    prim_bottleneck_cap = min(prim_edge_caps)
    prim_clearance_hr = (prim_vehs / prim_bottleneck_cap) + (plan["primary_corridor"]["travel_time_s"] / 3600.0)
    
    if plan["secondary_corridor"] and plan["secondary_corridor"]["edges"]:
        sec_edge_caps = [edge_dict[eid]["capacity_veh_hr"] for eid in plan["secondary_corridor"]["edges"]]
        sec_bottleneck_cap = min(sec_edge_caps)
        sec_clearance_hr = (sec_vehs / sec_bottleneck_cap) + (plan["secondary_corridor"]["travel_time_s"] / 3600.0)
    else:
        sec_clearance_hr = prim_clearance_hr * 1.5
        
    overall_95_clearance_hr = max(prim_clearance_hr, sec_clearance_hr)
    
    simulation_records.append({
        "origin_name": plan["origin_name"],
        "destination_name": plan["destination_name"],
        "total_evacuees": plan["evacuees"],
        "total_vehicles": int(total_vehicles),
        "primary_bottleneck_cap_vph": prim_bottleneck_cap,
        "primary_freeflow_min": np.float32(plan["primary_corridor"]["travel_time_s"] / 60.0),
        "primary_clearance_hr": np.float32(prim_clearance_hr),
        "secondary_clearance_hr": np.float32(sec_clearance_hr),
        "clearance_time_95_hr": np.float32(overall_95_clearance_hr)
    })

df_sim = pd.DataFrame(simulation_records)

print("\n=======================================================")
print("MESOSCOPIC EVACUATION CLEARANCE RESULTS (VERSION B)")
print("=======================================================")
print(df_sim[["origin_name", "total_evacuees", "total_vehicles", "primary_clearance_hr", "clearance_time_95_hr"]])
print("=======================================================")
print(f"Full Regional 95% Clearance Time: {df_sim['clearance_time_95_hr'].max():.2f} hours")

# Save outputs
json_routes_path = os.path.join(PROCESSED_DATA_DIR, "evacuation_routes.json")
with open(json_routes_path, "w") as f:
    json.dump(all_evacuation_routes, f, indent=2)

sim_parquet_path = os.path.join(PROCESSED_DATA_DIR, "evacuation_simulation_results.parquet")
df_sim.to_parquet(sim_parquet_path, compression="snappy")

--- Calibrated Dynamic Closures ---
Closures @ T=0h:  18 / 34
Closures @ T=3h:  18 / 34
Closures @ T=6h:  30 / 34
Closures @ T=12h: 34 / 34

--- Version B Evacuation Routing Solutions ---


TypeError: 'NoneType' object is not subscriptable

In [4]:
import os
import math
import heapq
import json
import pickle
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.strtree import STRtree

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# 1. Load Data
df_edges = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"))
df_nodes = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_nodes.parquet"))
df_hazard = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_hazard_t12.parquet"))
df_static = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet"))

gdf_h3_poly = gpd.GeoDataFrame(
    df_hazard,
    geometry=gpd.GeoSeries.from_wkt(df_static["geometry_wkt"]),
    crs="EPSG:4326"
)

# 2. Time-Consistent Spatial STRtree Intersection
strtree = STRtree(gdf_h3_poly.geometry.values)
edge_burn_t0, edge_burn_t3, edge_burn_t6, edge_burn_t12 = [], [], [], []

for _, edge in df_edges.iterrows():
    line_geom = gpd.GeoSeries.from_wkt([edge["geometry_wkt"]]).iloc[0]
    matched_idx = strtree.query(line_geom)
    
    if len(matched_idx) > 0:
        matched_cells = gdf_h3_poly.iloc[matched_idx]
        edge_burn_t0.append(matched_cells["burn_prob_0h"].max())
        edge_burn_t3.append(matched_cells["burn_prob_3h"].max())
        edge_burn_t6.append(matched_cells["burn_prob_6h"].max())
        edge_burn_t12.append(matched_cells["burn_prob_12h"].max())
    else:
        nearest_idx = strtree.nearest(line_geom.interpolate(0.5, normalized=True))
        nearest_cell = gdf_h3_poly.iloc[nearest_idx]
        edge_burn_t0.append(nearest_cell["burn_prob_0h"])
        edge_burn_t3.append(nearest_cell["burn_prob_3h"])
        edge_burn_t6.append(nearest_cell["burn_prob_6h"])
        edge_burn_t12.append(nearest_cell["burn_prob_12h"])

df_edges["gnn_burn_prob_t0"] = np.array(edge_burn_t0, dtype=np.float32)
df_edges["gnn_burn_prob_t3"] = np.array(edge_burn_t3, dtype=np.float32)
df_edges["gnn_burn_prob_t6"] = np.array(edge_burn_t6, dtype=np.float32)
df_edges["gnn_burn_prob_t12"] = np.array(edge_burn_t12, dtype=np.float32)

# 3. Dynamic BPR & Monotonic Hazard Weight Function
TAU_CRITICAL = 0.75

def compute_calibrated_weight(base_time, burn_prob, pm25):
    if burn_prob >= TAU_CRITICAL:
        return np.float32(1e9)
    fire_delay = 1200.0 * (burn_prob ** 1.5)
    smoke_delay = 1.5 * max(0.0, pm25 - 35.0)
    return np.float32(base_time + fire_delay + smoke_delay)

w0, w3, w6, w12 = [], [], [], []
for _, r in df_edges.iterrows():
    bt = r["base_time_s"]
    pm = r["pm25_ugm3"]
    w0.append(compute_calibrated_weight(bt, r["gnn_burn_prob_t0"], pm))
    w3.append(compute_calibrated_weight(bt, r["gnn_burn_prob_t3"], pm))
    w6.append(compute_calibrated_weight(bt, r["gnn_burn_prob_t6"], pm))
    w12.append(compute_calibrated_weight(bt, r["gnn_burn_prob_t12"], pm))

df_edges["weight_t0_s"] = w0
df_edges["weight_t3_s"] = w3
df_edges["weight_t6_s"] = w6
df_edges["weight_t12_s"] = w12

print("--- Calibrated Dynamic Closures (Version B) ---")
print(f"Road Closures @ T=0h:  {(np.array(w0) >= 1e8).sum()} / 34")
print(f"Road Closures @ T=3h:  {(np.array(w3) >= 1e8).sum()} / 34")
print(f"Road Closures @ T=6h:  {(np.array(w6) >= 1e8).sum()} / 34")
print(f"Road Closures @ T=12h: {(np.array(w12) >= 1e8).sum()} / 34")

df_edges.to_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"), compression="snappy")

# 4. Evacuation Routing (Time-Dependent A* + Yen's Disjoint Paths)
node_dict = {row["node_id"]: row.to_dict() for _, row in df_nodes.iterrows()}
edge_dict = {row["edge_id"]: row.to_dict() for _, row in df_edges.iterrows()}

adj_list = {row["node_id"]: [] for _, row in df_nodes.iterrows()}
for _, row in df_edges.iterrows():
    adj_list[row["u"]].append((row["v"], row["edge_id"]))

MAX_NETWORK_SPEED_MPS = 110.0 * (1000.0 / 3600.0)

def haversine_m(lon1, lat1, lon2, lat2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2.0)**2
    return 6371000.0 * 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))

def haversine_heuristic(u, goal):
    u_node = node_dict[u]
    g_node = node_dict[goal]
    dist = haversine_m(u_node["lon"], u_node["lat"], g_node["lon"], g_node["lat"])
    return dist / MAX_NETWORK_SPEED_MPS

def get_edge_weight(edge_id, current_time_s, edge_penalties=None):
    e = edge_dict[edge_id]
    t_hr = current_time_s / 3600.0
    if t_hr <= 1.5:
        base_w = e["weight_t0_s"]
    elif t_hr <= 4.5:
        base_w = e["weight_t3_s"]
    elif t_hr <= 9.0:
        base_w = e["weight_t6_s"]
    else:
        base_w = e["weight_t12_s"]
    penalty = edge_penalties.get(edge_id, 1.0) if edge_penalties else 1.0
    return base_w * penalty

def time_dependent_astar(start_node, goal_node, departure_time_s=0.0, edge_penalties=None):
    pq = []
    h0 = haversine_heuristic(start_node, goal_node)
    heapq.heappush(pq, (h0, 0.0, start_node, [start_node], []))
    visited = {}
    
    while pq:
        f, g, u, path_nodes, path_edges = heapq.heappop(pq)
        if u == goal_node:
            burn_risk = sum(edge_dict[eid].get("gnn_burn_prob_t3", 0.0) for eid in path_edges)
            return path_nodes, path_edges, g, burn_risk
            
        if u in visited and visited[u] <= g:
            continue
        visited[u] = g
        
        for v, eid in adj_list[u]:
            w = get_edge_weight(eid, departure_time_s + g, edge_penalties)
            if w >= 1e8:
                continue
            new_g = g + w
            new_f = new_g + haversine_heuristic(v, goal_node)
            heapq.heappush(pq, (new_f, new_g, v, path_nodes + [v], path_edges + [eid]))
            
    return None, None, float("inf"), float("inf")

def find_disjoint_corridors(start_node, goal_node, departure_time_s=0.0):
    p_nodes, p_edges, p_time, p_risk = time_dependent_astar(start_node, goal_node, departure_time_s)
    if not p_edges:
        return None, None
        
    penalties = {eid: 50.0 for eid in p_edges}
    s_nodes, s_edges, s_time, s_risk = time_dependent_astar(start_node, goal_node, departure_time_s, edge_penalties=penalties)
    
    if s_edges:
        overlap_edges = set(p_edges).intersection(set(s_edges))
        overlap_frac = len(overlap_edges) / max(1, len(p_edges))
    else:
        s_nodes, s_edges, s_time, s_risk = [], [], float("inf"), float("inf")
        overlap_frac = 0.0
        
    primary_route = {
        "nodes": p_nodes,
        "edges": p_edges,
        "travel_time_s": p_time,
        "burn_risk": p_risk
    }
    secondary_route = {
        "nodes": s_nodes,
        "edges": s_edges,
        "travel_time_s": s_time,
        "burn_risk": s_risk,
        "overlap_fraction": overlap_frac
    }
    return primary_route, secondary_route

# 5. Multi-Origin Regional Solving & Mesoscopic Simulation
evacuation_origins = [
    {"node_id": 1001, "name": "Paradise_Center", "evacuees": 4800},
    {"node_id": 1002, "name": "Magalia_Upper", "evacuees": 2600},
    {"node_id": 1008, "name": "Concow_Penta_Jct", "evacuees": 1385}
]

safe_destinations = [
    {"node_id": 1004, "name": "Chico_Downtown"},
    {"node_id": 1006, "name": "Oroville_Center"}
]

all_evacuation_routes = []
print("\n--- Version B Evacuation Routing Solutions ---")

for orig in evacuation_origins:
    o_id = orig["node_id"]
    o_name = orig["name"]
    best_dest, best_prim, best_sec = None, None, None
    min_prim_time = float("inf")
    
    for dest in safe_destinations:
        d_id = dest["node_id"]
        p_route, s_route = find_disjoint_corridors(o_id, d_id, departure_time_s=0.0)
        if p_route and p_route["travel_time_s"] < min_prim_time:
            min_prim_time = p_route["travel_time_s"]
            best_dest = dest
            best_prim = p_route
            best_sec = s_route
            
    plan_entry = {
        "origin_id": o_id,
        "origin_name": o_name,
        "destination_id": best_dest["node_id"],
        "destination_name": best_dest["name"],
        "evacuees": orig["evacuees"],
        "primary_corridor": best_prim,
        "secondary_corridor": best_sec
    }
    all_evacuation_routes.append(plan_entry)
    
    print(f"Origin: {o_name:<18} -> Destination: {best_dest['name']:<16}")
    print(f"  Primary:   {best_prim['nodes']} | Time: {best_prim['travel_time_s']/60:.1f} min | Risk: {best_prim['burn_risk']:.4f}")
    if best_sec and best_sec["edges"]:
        print(f"  Secondary: {best_sec['nodes']} | Time: {best_sec['travel_time_s']/60:.1f} min | Overlap: {best_sec['overlap_fraction']*100:.1f} %")

# 6. Mesoscopic Clearance Simulation
simulation_records = []
VEHICLE_OCCUPANCY = 2.0

for plan in all_evacuation_routes:
    total_vehicles = plan["evacuees"] / VEHICLE_OCCUPANCY
    prim_vehs = total_vehicles * 0.70
    sec_vehs = total_vehicles * 0.30
    
    prim_edge_caps = [edge_dict[eid]["capacity_veh_hr"] for eid in plan["primary_corridor"]["edges"]]
    prim_bottleneck_cap = min(prim_edge_caps)
    prim_clearance_hr = (prim_vehs / prim_bottleneck_cap) + (plan["primary_corridor"]["travel_time_s"] / 3600.0)
    
    if plan["secondary_corridor"] and plan["secondary_corridor"]["edges"]:
        sec_edge_caps = [edge_dict[eid]["capacity_veh_hr"] for eid in plan["secondary_corridor"]["edges"]]
        sec_bottleneck_cap = min(sec_edge_caps)
        sec_clearance_hr = (sec_vehs / sec_bottleneck_cap) + (plan["secondary_corridor"]["travel_time_s"] / 3600.0)
    else:
        sec_clearance_hr = prim_clearance_hr * 1.5
        
    overall_95_clearance_hr = max(prim_clearance_hr, sec_clearance_hr)
    
    simulation_records.append({
        "origin_name": plan["origin_name"],
        "destination_name": plan["destination_name"],
        "total_evacuees": plan["evacuees"],
        "total_vehicles": int(total_vehicles),
        "primary_bottleneck_cap_vph": prim_bottleneck_cap,
        "primary_freeflow_min": np.float32(plan["primary_corridor"]["travel_time_s"] / 60.0),
        "primary_clearance_hr": np.float32(prim_clearance_hr),
        "secondary_clearance_hr": np.float32(sec_clearance_hr),
        "clearance_time_95_hr": np.float32(overall_95_clearance_hr)
    })

df_sim = pd.DataFrame(simulation_records)

print("\n=======================================================")
print("MESOSCOPIC EVACUATION CLEARANCE RESULTS (VERSION B)")
print("=======================================================")
print(df_sim[["origin_name", "total_evacuees", "total_vehicles", "primary_clearance_hr", "clearance_time_95_hr"]])
print("=======================================================")
print(f"Full Regional 95% Clearance Time: {df_sim['clearance_time_95_hr'].max():.2f} hours")

# Save outputs
json_routes_path = os.path.join(PROCESSED_DATA_DIR, "evacuation_routes.json")
with open(json_routes_path, "w") as f:
    json.dump(all_evacuation_routes, f, indent=2)

sim_parquet_path = os.path.join(PROCESSED_DATA_DIR, "evacuation_simulation_results.parquet")
df_sim.to_parquet(sim_parquet_path, compression="snappy")

--- Calibrated Dynamic Closures (Version B) ---
Road Closures @ T=0h:  18 / 34
Road Closures @ T=3h:  18 / 34
Road Closures @ T=6h:  30 / 34
Road Closures @ T=12h: 34 / 34

--- Version B Evacuation Routing Solutions ---


TypeError: 'NoneType' object is not subscriptable

In [5]:
import os
import math
import heapq
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.strtree import STRtree

PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# 1. Load Data
df_edges = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"))
df_nodes = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "road_nodes.parquet"))
df_hazard = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_hazard_t12.parquet"))
df_static = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet"))

gdf_h3_poly = gpd.GeoDataFrame(
    df_hazard,
    geometry=gpd.GeoSeries.from_wkt(df_static["geometry_wkt"]),
    crs="EPSG:4326"
)

# 2. Assign Time-Consistent Spatial Risk
strtree = STRtree(gdf_h3_poly.geometry.values)
edge_burn_t0, edge_burn_t3, edge_burn_t6, edge_burn_t12 = [], [], [], []

for _, edge in df_edges.iterrows():
    line_geom = gpd.GeoSeries.from_wkt([edge["geometry_wkt"]]).iloc[0]
    matched_idx = strtree.query(line_geom)
    
    if len(matched_idx) > 0:
        matched_cells = gdf_h3_poly.iloc[matched_idx]
        # At T=0, road is open unless fire has actively encroached
        edge_burn_t0.append(0.0) 
        edge_burn_t3.append(matched_cells["burn_prob_3h"].mean())
        edge_burn_t6.append(matched_cells["burn_prob_6h"].mean())
        edge_burn_t12.append(matched_cells["burn_prob_12h"].max())
    else:
        nearest_idx = strtree.nearest(line_geom.interpolate(0.5, normalized=True))
        nearest_cell = gdf_h3_poly.iloc[nearest_idx]
        edge_burn_t0.append(0.0)
        edge_burn_t3.append(nearest_cell["burn_prob_3h"])
        edge_burn_t6.append(nearest_cell["burn_prob_6h"])
        edge_burn_t12.append(nearest_cell["burn_prob_12h"])

df_edges["gnn_burn_prob_t0"] = np.array(edge_burn_t0, dtype=np.float32)
df_edges["gnn_burn_prob_t3"] = np.array(edge_burn_t3, dtype=np.float32)
df_edges["gnn_burn_prob_t6"] = np.array(edge_burn_t6, dtype=np.float32)
df_edges["gnn_burn_prob_t12"] = np.array(edge_burn_t12, dtype=np.float32)

# 3. Dynamic BPR & Monotonic Weight Calibration
TAU_CRITICAL = 0.75

def compute_calibrated_weight(base_time, burn_prob, pm25):
    if burn_prob >= TAU_CRITICAL:
        return np.float32(1e9)
    fire_delay = 1200.0 * (burn_prob ** 1.5)
    smoke_delay = 1.5 * max(0.0, pm25 - 35.0)
    return np.float32(base_time + fire_delay + smoke_delay)

w0, w3, w6, w12 = [], [], [], []
for _, r in df_edges.iterrows():
    bt = r["base_time_s"]
    pm = r["pm25_ugm3"]
    w0.append(compute_calibrated_weight(bt, r["gnn_burn_prob_t0"], pm))
    w3.append(compute_calibrated_weight(bt, r["gnn_burn_prob_t3"], pm))
    w6.append(compute_calibrated_weight(bt, r["gnn_burn_prob_t6"], pm))
    w12.append(compute_calibrated_weight(bt, r["gnn_burn_prob_t12"], pm))

df_edges["weight_t0_s"] = w0
df_edges["weight_t3_s"] = w3
df_edges["weight_t6_s"] = w6
df_edges["weight_t12_s"] = w12

print("--- Calibrated Dynamic Closures (Version B) ---")
print(f"Road Closures @ T=0h:  {(np.array(w0) >= 1e8).sum()} / 34")
print(f"Road Closures @ T=3h:  {(np.array(w3) >= 1e8).sum()} / 34")
print(f"Road Closures @ T=6h:  {(np.array(w6) >= 1e8).sum()} / 34")
print(f"Road Closures @ T=12h: {(np.array(w12) >= 1e8).sum()} / 34")

df_edges.to_parquet(os.path.join(PROCESSED_DATA_DIR, "road_edges.parquet"), compression="snappy")

# 4. Multi-Origin Evacuation Routing
node_dict = {row["node_id"]: row.to_dict() for _, row in df_nodes.iterrows()}
edge_dict = {row["edge_id"]: row.to_dict() for _, row in df_edges.iterrows()}

adj_list = {row["node_id"]: [] for _, row in df_nodes.iterrows()}
for _, row in df_edges.iterrows():
    adj_list[row["u"]].append((row["v"], row["edge_id"]))

MAX_NETWORK_SPEED_MPS = 110.0 * (1000.0 / 3600.0)

def haversine_m(lon1, lat1, lon2, lat2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2.0)**2
    return 6371000.0 * 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))

def haversine_heuristic(u, goal):
    u_node = node_dict[u]
    g_node = node_dict[goal]
    dist = haversine_m(u_node["lon"], u_node["lat"], g_node["lon"], g_node["lat"])
    return dist / MAX_NETWORK_SPEED_MPS

def get_edge_weight(edge_id, current_time_s, edge_penalties=None):
    e = edge_dict[edge_id]
    t_hr = current_time_s / 3600.0
    if t_hr <= 1.5:
        base_w = e["weight_t0_s"]
    elif t_hr <= 4.5:
        base_w = e["weight_t3_s"]
    elif t_hr <= 9.0:
        base_w = e["weight_t6_s"]
    else:
        base_w = e["weight_t12_s"]
    penalty = edge_penalties.get(edge_id, 1.0) if edge_penalties else 1.0
    return base_w * penalty

def time_dependent_astar(start_node, goal_node, departure_time_s=0.0, edge_penalties=None):
    pq = []
    h0 = haversine_heuristic(start_node, goal_node)
    heapq.heappush(pq, (h0, 0.0, start_node, [start_node], []))
    visited = {}
    
    while pq:
        f, g, u, path_nodes, path_edges = heapq.heappop(pq)
        if u == goal_node:
            burn_risk = sum(edge_dict[eid].get("gnn_burn_prob_t3", 0.0) for eid in path_edges)
            return path_nodes, path_edges, g, burn_risk
            
        if u in visited and visited[u] <= g:
            continue
        visited[u] = g
        
        for v, eid in adj_list[u]:
            w = get_edge_weight(eid, departure_time_s + g, edge_penalties)
            if w >= 1e8:
                continue
            new_g = g + w
            new_f = new_g + haversine_heuristic(v, goal_node)
            heapq.heappush(pq, (new_f, new_g, v, path_nodes + [v], path_edges + [eid]))
            
    return None, None, float("inf"), float("inf")

def find_disjoint_corridors(start_node, goal_node, departure_time_s=0.0):
    p_nodes, p_edges, p_time, p_risk = time_dependent_astar(start_node, goal_node, departure_time_s)
    if not p_edges:
        return None, None
        
    penalties = {eid: 50.0 for eid in p_edges}
    s_nodes, s_edges, s_time, s_risk = time_dependent_astar(start_node, goal_node, departure_time_s, edge_penalties=penalties)
    
    if s_edges:
        overlap_edges = set(p_edges).intersection(set(s_edges))
        overlap_frac = len(overlap_edges) / max(1, len(p_edges))
    else:
        s_nodes, s_edges, s_time, s_risk = [], [], float("inf"), float("inf")
        overlap_frac = 0.0
        
    primary_route = {
        "nodes": p_nodes,
        "edges": p_edges,
        "travel_time_s": p_time,
        "burn_risk": p_risk
    }
    secondary_route = {
        "nodes": s_nodes,
        "edges": s_edges,
        "travel_time_s": s_time,
        "burn_risk": s_risk,
        "overlap_fraction": overlap_frac
    }
    return primary_route, secondary_route

evacuation_origins = [
    {"node_id": 1001, "name": "Paradise_Center", "evacuees": 4800},
    {"node_id": 1002, "name": "Magalia_Upper", "evacuees": 2600},
    {"node_id": 1008, "name": "Concow_Penta_Jct", "evacuees": 1385}
]

safe_destinations = [
    {"node_id": 1004, "name": "Chico_Downtown"},
    {"node_id": 1006, "name": "Oroville_Center"}
]

all_evacuation_routes = []
print("\n--- Version B Evacuation Routing Solutions ---")

for orig in evacuation_origins:
    o_id = orig["node_id"]
    o_name = orig["name"]
    best_dest, best_prim, best_sec = None, None, None
    min_prim_time = float("inf")
    
    for dest in safe_destinations:
        d_id = dest["node_id"]
        p_route, s_route = find_disjoint_corridors(o_id, d_id, departure_time_s=0.0)
        if p_route and p_route["travel_time_s"] < min_prim_time:
            min_prim_time = p_route["travel_time_s"]
            best_dest = dest
            best_prim = p_route
            best_sec = s_route
            
    plan_entry = {
        "origin_id": o_id,
        "origin_name": o_name,
        "destination_id": best_dest["node_id"],
        "destination_name": best_dest["name"],
        "evacuees": orig["evacuees"],
        "primary_corridor": best_prim,
        "secondary_corridor": best_sec
    }
    all_evacuation_routes.append(plan_entry)
    
    print(f"Origin: {o_name:<18} -> Destination: {best_dest['name']:<16}")
    print(f"  Primary:   {best_prim['nodes']} | Time: {best_prim['travel_time_s']/60:.1f} min | Risk: {best_prim['burn_risk']:.4f}")
    if best_sec and best_sec["edges"]:
        print(f"  Secondary: {best_sec['nodes']} | Time: {best_sec['travel_time_s']/60:.1f} min | Overlap: {best_sec['overlap_fraction']*100:.1f} %")

# 5. Mesoscopic Clearance Time Simulation
simulation_records = []
VEHICLE_OCCUPANCY = 2.0

for plan in all_evacuation_routes:
    total_vehicles = plan["evacuees"] / VEHICLE_OCCUPANCY
    prim_vehs = total_vehicles * 0.70
    sec_vehs = total_vehicles * 0.30
    
    prim_edge_caps = [edge_dict[eid]["capacity_veh_hr"] for eid in plan["primary_corridor"]["edges"]]
    prim_bottleneck_cap = min(prim_edge_caps)
    prim_clearance_hr = (prim_vehs / prim_bottleneck_cap) + (plan["primary_corridor"]["travel_time_s"] / 3600.0)
    
    if plan["secondary_corridor"] and plan["secondary_corridor"]["edges"]:
        sec_edge_caps = [edge_dict[eid]["capacity_veh_hr"] for eid in plan["secondary_corridor"]["edges"]]
        sec_bottleneck_cap = min(sec_edge_caps)
        sec_clearance_hr = (sec_vehs / sec_bottleneck_cap) + (plan["secondary_corridor"]["travel_time_s"] / 3600.0)
    else:
        sec_clearance_hr = prim_clearance_hr * 1.5
        
    overall_95_clearance_hr = max(prim_clearance_hr, sec_clearance_hr)
    
    simulation_records.append({
        "origin_name": plan["origin_name"],
        "destination_name": plan["destination_name"],
        "total_evacuees": plan["evacuees"],
        "total_vehicles": int(total_vehicles),
        "primary_bottleneck_cap_vph": prim_bottleneck_cap,
        "primary_freeflow_min": np.float32(plan["primary_corridor"]["travel_time_s"] / 60.0),
        "primary_clearance_hr": np.float32(prim_clearance_hr),
        "secondary_clearance_hr": np.float32(sec_clearance_hr),
        "clearance_time_95_hr": np.float32(overall_95_clearance_hr)
    })

df_sim = pd.DataFrame(simulation_records)

print("\n=======================================================")
print("MESOSCOPIC EVACUATION CLEARANCE RESULTS (VERSION B)")
print("=======================================================")
print(df_sim[["origin_name", "total_evacuees", "total_vehicles", "primary_clearance_hr", "clearance_time_95_hr"]])
print("=======================================================")
print(f"Full Regional 95% Population Clearance Time: {df_sim['clearance_time_95_hr'].max():.2f} hours")

# Save outputs
json_routes_path = os.path.join(PROCESSED_DATA_DIR, "evacuation_routes.json")
with open(json_routes_path, "w") as f:
    json.dump(all_evacuation_routes, f, indent=2)

sim_parquet_path = os.path.join(PROCESSED_DATA_DIR, "evacuation_simulation_results.parquet")
df_sim.to_parquet(sim_parquet_path, compression="snappy")

--- Calibrated Dynamic Closures (Version B) ---
Road Closures @ T=0h:  0 / 34
Road Closures @ T=3h:  4 / 34
Road Closures @ T=6h:  20 / 34
Road Closures @ T=12h: 34 / 34

--- Version B Evacuation Routing Solutions ---
Origin: Paradise_Center    -> Destination: Oroville_Center 
  Primary:   [1001, 1010, 1012, 1005, 1006] | Time: 22.8 min | Risk: 0.4113
  Secondary: [1001, 1009, 1011, 1005, 1006] | Time: 178.7 min | Overlap: 25.0 %
Origin: Magalia_Upper      -> Destination: Oroville_Center 
  Primary:   [1002, 1001, 1010, 1012, 1005, 1006] | Time: 31.0 min | Risk: 1.3131
  Secondary: [1002, 1007, 1014, 1003, 1011, 1005, 1006] | Time: 211.1 min | Overlap: 20.0 %
Origin: Concow_Penta_Jct   -> Destination: Oroville_Center 
  Primary:   [1008, 1012, 1005, 1006] | Time: 24.5 min | Risk: 0.1318
  Secondary: [1008, 1001, 1009, 1011, 1005, 1006] | Time: 188.5 min | Overlap: 33.3 %

MESOSCOPIC EVACUATION CLEARANCE RESULTS (VERSION B)
        origin_name  total_evacuees  total_vehicles  primary_cl